# HearAI — Phase 1 : Dataset et prétraitement audio (ESC-50)

## Objectif
Préparer les sons pour le Deep Learning. À la fin, chaque fichier audio sera transformé en **Mel spectrogram** de taille identique et sauvegardé dans `sound/data/processed/esc50_mel_dataset.npz` pour la Phase 2.

**Dossier commun Google Drive :** `MyDrive/HearAI/sound/`. Les deux notebooks utilisent ce même emplacement.

Pipeline : fichier WAV → mono → 16 kHz → 5 secondes → Mel spectrogram → sauvegarde.

**À savoir :** ESC-50 contient 2 000 sons, 50 classes et 40 exemples par classe. C'est un dataset généraliste : il ne couvre pas tous les dangers possibles.


## 1. Préparer l'environnement

Ce notebook est prévu pour Google Colab. Pour accélérer le traitement, sélectionnez si possible `Exécution > Modifier le type d'exécution > GPU`. Le prétraitement reste possible sur CPU.

In [ ]:
!pip -q install librosa soundfile pandas numpy matplotlib seaborn scikit-learn tqdm

In [ ]:
import random, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import soundfile as sf
from tqdm.auto import tqdm

# 1) Monter Google Drive (à faire dans chaque notebook Colab)
from google.colab import drive
drive.mount('/content/drive')

# 2) Architecture commune HearAI/sound dans Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/HearAI/sound')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

# ESC-50 sera téléchargé dans data/raw/ESC-50
DATASET_DIR = RAW_DIR / 'ESC-50'
AUDIO_DIR = DATASET_DIR / 'audio'
META_PATH = DATASET_DIR / 'meta' / 'esc50.csv'
OUTPUT_DIR = PROCESSED_DIR

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Projet :', PROJECT_DIR)
print('Dataset brut :', DATASET_DIR)
print('Sortie prétraitée :', OUTPUT_DIR)


## 2. Télécharger ESC-50 dans Google Drive

Le dépôt officiel sera placé dans `sound/data/raw/ESC-50/`. Le téléchargement n'est effectué que si ce dossier n'existe pas déjà.

Le dataset prétraité sera ensuite enregistré dans `sound/data/processed/esc50_mel_dataset.npz`, pour être lu directement par le notebook de Phase 2.


In [ ]:
if not DATASET_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/karolpiczak/ESC-50.git", str(DATASET_DIR)],
        check=True
    )

print("CSV présent :", META_PATH.exists())
print("Fichiers WAV :", len(list(AUDIO_DIR.glob("*.wav"))))

## 3. Explorer les métadonnées

Le fichier CSV contient notamment `filename` (nom du son), `category` (classe) et `fold` (groupe officiel). Le fold servira à séparer train, validation et test dans la Phase 2.

In [ ]:
metadata = pd.read_csv(META_PATH)
print("Dimensions :", metadata.shape)
display(metadata.head())
print("\\nTypes :")
display(metadata.dtypes)
print("\\nValeurs manquantes :")
display(metadata.isna().sum())
print("\\nDoublons filename :", metadata["filename"].duplicated().sum())
print("Classes :", metadata["category"].nunique())
print("Folds :", sorted(metadata["fold"].unique()))

## 4. Vérifier l'équilibre des classes

ESC-50 contient normalement 40 fichiers par classe. Une distribution anormale peut signaler un problème de données.

In [ ]:
counts = metadata["category"].value_counts().sort_values()
display(counts.to_frame("nombre_de_fichiers"))
plt.figure(figsize=(12, 10))
counts.plot(kind="barh")
plt.title("Nombre de fichiers par classe")
plt.xlabel("Nombre de fichiers")
plt.tight_layout()
plt.show()

## 5. Comprendre un signal audio

Un signal audio est une suite de valeurs numériques. La fréquence d'échantillonnage indique le nombre de valeurs par seconde. La durée se calcule par `nombre_de_valeurs / fréquence`.

On examine un son avant transformation.

In [ ]:
row = metadata.iloc[0]
path = AUDIO_DIR / row["filename"]
audio_original, sr_original = sf.read(path)
if audio_original.ndim == 2:
    audio_original = audio_original.mean(axis=1)

print("Fichier :", row["filename"])
print("Classe :", row["category"])
print("Fréquence :", sr_original, "Hz")
print("Durée :", len(audio_original) / sr_original, "secondes")

t = np.arange(len(audio_original)) / sr_original
plt.figure(figsize=(13, 3))
plt.plot(t, audio_original)
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.title("Forme d'onde du signal")
plt.show()

from IPython.display import Audio, display
display(Audio(audio_original, rate=sr_original))

## 6. Définir les paramètres de prétraitement

| Paramètre | Valeur | Signification |
|---|---:|---|
| Sample rate | 16 000 Hz | fréquence commune |
| Durée | 5 s | longueur fixe |
| `n_mels` | 128 | nombre de bandes Mel |
| `n_fft` | 2048 | taille de fenêtre pour l'analyse fréquentielle |
| `hop_length` | 512 | décalage entre fenêtres |

Le **Mel spectrogram** représente l'énergie des fréquences au cours du temps sur une échelle Mel. On le traite comme une image à un canal pour le CNN.

In [ ]:
SAMPLE_RATE = 16000
DURATION = 5
TARGET_LENGTH = SAMPLE_RATE * DURATION
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

def load_and_standardize_audio(file_path):
    # Charge en mono et rééchantillonne à 16 kHz.
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    # Tous les signaux doivent faire exactement 5 secondes.
    if len(audio) < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - len(audio)))
    else:
        audio = audio[:TARGET_LENGTH]
    return audio.astype(np.float32)

def audio_to_log_mel(audio):
    # Calcul du spectrogramme de puissance sur l'échelle Mel.
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0
    )
    # Conversion en décibels, puis normalisation entre 0 et 1.
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return np.clip((mel_db + 80.0) / 80.0, 0.0, 1.0).astype(np.float32)

## 7. Tester sur un seul fichier

On valide les fonctions avant de traiter tout le dataset. Avec ces paramètres, le signal audio doit contenir 80 000 valeurs (16 000 × 5). Le Mel spectrogram a normalement la forme `(128, 157)`.

In [ ]:
audio_test = load_and_standardize_audio(path)
mel_test = audio_to_log_mel(audio_test)

print("Audio :", audio_test.shape, audio_test.dtype)
print("Mel :", mel_test.shape, mel_test.dtype)
print("Min/max :", mel_test.min(), mel_test.max())
print("NaN :", np.isnan(mel_test).any(), "| Inf :", np.isinf(mel_test).any())

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    mel_test, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
    x_axis="time", y_axis="mel"
)
plt.colorbar()
plt.title("Mel spectrogram normalisé")
plt.show()

## 8. Transformer les 2 000 fichiers

Pour chaque fichier, on applique exactement les mêmes opérations. On conserve la classe et le fold associés à chaque spectrogramme. Cette boucle peut prendre quelques minutes.

In [ ]:
X_list, y_list, fold_list, filename_list = [], [], [], []

for r in tqdm(metadata.itertuples(index=False), total=len(metadata)):
    audio = load_and_standardize_audio(AUDIO_DIR / r.filename)
    mel = audio_to_log_mel(audio)
    X_list.append(mel)
    y_list.append(r.category)
    fold_list.append(int(r.fold))
    filename_list.append(r.filename)

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list)
folds = np.array(fold_list, dtype=np.int64)
filenames = np.array(filename_list)

print("X :", X.shape)
print("y :", y.shape)
print("folds :", folds.shape)

## 9. Contrôler les données transformées

Avant l'entraînement, on contrôle les dimensions, les valeurs manquantes, les valeurs infinies et les bornes de normalisation.

In [ ]:
print("Exemples :", len(X))
print("Classes :", len(np.unique(y)))
print("NaN :", np.isnan(X).any())
print("Inf :", np.isinf(X).any())
print("Min/max :", float(X.min()), float(X.max()))
print("Exemples par fold :")
display(pd.Series(folds).value_counts().sort_index().to_frame("nombre"))

## 10. Sauvegarder le résultat

Le fichier `.npz` contient les spectrogrammes, les labels, les folds et les noms des fichiers. La Phase 2 lira ce fichier : il n'est donc pas nécessaire de refaire le prétraitement à chaque entraînement.

Attention : `/content` est temporaire dans Colab. Téléchargez le fichier ou copiez-le dans Google Drive pour le conserver.

In [ ]:
processed_path = OUTPUT_DIR / "esc50_mel_dataset.npz"
np.savez_compressed(
    processed_path, X=X, y=y, folds=folds, filenames=filenames
)
print("Sauvegardé :", processed_path)
print("Taille (Mo) :", round(processed_path.stat().st_size / (1024**2), 2))

In [ ]:
print('Le dataset est conservé dans Google Drive :')
print(processed_path)
print('Fichier présent :', processed_path.exists())
print('Taille (Mo) :', round(processed_path.stat().st_size / (1024**2), 2))


## Bilan Phase 1

Résultats attendus :
- `X` : `(2000, 128, 157)` ;
- `y` : `(2000,)` ;
- 50 classes ;
- 5 folds ;
- fichier `esc50_mel_dataset.npz`.

La Phase 1 est terminée lorsque le fichier NPZ est téléchargé. Passez ensuite au notebook Phase 2.